In [ ]:
import glob
import os

import pandas as pd
import torch

os.chdir("..")
os.getcwd()

In [ ]:
# Check how do aef and tessera files overlap

paths = glob.glob("data/s2bms/eo/aef/*.tif")
aef = []
for p in paths:
    aef.append(os.path.basename(p).split(".")[0].split("-")[-1])


paths = glob.glob("data/s2bms/eo/tessera/*.npy")
tsr = []
for p in paths:
    tsr.append(os.path.basename(p).split(".")[0].split("-")[-1])

common = list(set(tsr) & set(aef))

tsr_left = list(set(tsr) - set(common))  # None
aef_left = list(set(aef) - set(common))  # 12

print("Tessera:", tsr_left)
print("AlphaEarth:", aef_left)

In [ ]:
# Load in original datasplit file
pth = "data/s2bms/splits/split_indices_s2bms_2024-08-14-1459.pth"
split_indices = torch.load(pth, weights_only=False)

In [ ]:
# Print data split sizes
s = 0
print("Before cleaning:")
for k, v in split_indices.items():
    if not k == "clusters":
        print(k, len(v))
        s += len(v)

In [ ]:
# Filter out remaining aef but missing tessera tiles form val and test splits

aef_left_str = [f"UKBMS_loc-{idx}" for idx in aef_left]

split_indices["val_indices"] = split_indices["val_indices"][
    ~split_indices["val_indices"].isin(aef_left_str)
]
split_indices["test_indices"] = split_indices["test_indices"][
    ~split_indices["test_indices"].isin(aef_left_str)
]

print("After test/val cleaning:")
for k, v in split_indices.items():
    if not k == "clusters":
        print(k, len(v))
        s += len(v)

if os.path.exists(
    "data/s2bms/splits/split_indices_s2bms_2024-08-14-1459_union_aef_tsr_val_test.pth"
):
    print("Already saved")
else:
    torch.save(
        split_indices,
        "data/s2bms/splits/split_indices_s2bms_2024-08-14-1459_union_aef_tsr_val_test.pth",
    )

In [ ]:
# Filter out remaining aef but missing tessera tiles form train splits

split_indices["train_indices"] = split_indices["train_indices"][
    ~split_indices["train_indices"].isin(aef_left_str)
]

print("After test/val/train cleaning:")
for k, v in split_indices.items():
    if not k == "clusters":
        print(k, len(v))
        s += len(v)

if os.path.exists(
    "data/s2bms/splits/split_indices_s2bms_2024-08-14-1459_union_aef_tsr_val_test_train.pth"
):
    print("Already saved")
else:
    torch.save(
        split_indices,
        "data/s2bms/splits/split_indices_s2bms_2024-08-14-1459_union_aef_tsr_val_test_train.pth",
    )

In [ ]:
# Visualise data split
df = pd.read_csv("data/s2bms/model_ready_s2bms.csv")

train_idx = split_indices["train_indices"]
train_df = df[df["name_loc"].isin(map(str, train_idx))]


val_idx = split_indices["val_indices"]
val_df = df[df["name_loc"].isin(map(str, val_idx))]

test_idx = split_indices["test_indices"]
test_df = df[df["name_loc"].isin(map(str, test_idx))]

In [ ]:
import folium

m = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=3)

# Train = blue
for _, row in train_df.iterrows():
    folium.CircleMarker(location=[row.lat, row.lon], radius=2, color="blue", fill=True).add_to(m)

# Val = orange
for _, row in val_df.iterrows():
    folium.CircleMarker(location=[row.lat, row.lon], radius=2, color="orange", fill=True).add_to(m)

# Test = red
for _, row in test_df.iterrows():
    folium.CircleMarker(location=[row.lat, row.lon], radius=2, color="red", fill=True).add_to(m)

m